In [ ]:
pip install wandb

In [ ]:
!pip install wandb -qU

In [ ]:
# Log in to your W&B account
import wandb
import random
import math



wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [ ]:
 # Replace 'YOUR_API_KEY' with your actual W&B API key

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [ ]:
import wandb
wandb.init(
    project="NALAPRO-Project",
    config={
        "learning_rate": 2e-5,
        "architecture": "BERT-Base-Uncased",
        "dataset": "20Newsgroups"
    }
)

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split

# Download required NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab') # Added this line to download the missing resource

def load_and_preprocess_data():
    """
    Loads 20Newsgroups dataset filtering headers, footers, and quotes.
    Performs lowercasing, punctuation stripping, and stopword removal.
    """
    print("Loading 20Newsgroups data...")
    # Fetch all data to create an explicit, identical train/test split across tasks
    raw_data = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))

    stop_words = set(stopwords.words('english'))
    cleaned_texts = []
    tokenized_texts = []

    print("Preprocessing corpus...")
    for text in raw_data.data:
        # Convert to lowercase and clean non-alphabetic strings
        text_lower = text.lower()
        words = word_tokenize(text_lower)

        # Keep alphabetic tokens and drop stopwords/short tokens
        cleaned_tokens = [w for w in words if w.isalpha() and w not in stop_words and len(w) > 1]

        tokenized_texts.append(cleaned_tokens)
        cleaned_texts.append(" ".join(cleaned_tokens))

    # Create fixed structural splits (80% Train, 20% Test)
    indices = range(len(raw_data.data))
    train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=raw_data.target)

    return {
        "raw_labels": raw_data.target,
        "target_names": raw_data.target_names,
        "cleaned_texts": cleaned_texts,       # For TF-IDF and Transformers
        "tokenized_texts": tokenized_texts,   # For Word2Vec
        "splits": (train_idx, test_idx)
    }

data_bundle = load_and_preprocess_data()
print(f"Data engine ready. Total processed documents: {len(data_bundle['cleaned_texts'])}")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Loading 20Newsgroups data...
Preprocessing corpus...
Data engine ready. Total processed documents: 18846


In [ ]:
!pip install gensim -qU

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 46.6 MB/s eta 0:00:00


In [ ]:
"""
NALAPRO Project - Task 1: Classic NLP Vectorizations & FFN Classifier
"""
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score, f1_score
import gensim
from gensim.models import Word2Vec
import wandb

# Removed: from data_loader import load_and_preprocess_data

# Ensure reproducibility
torch.manual_seed(42)
np.random.seed(42)

# --- Neural Network Architecture ---
class FFNClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_classes=20):
        super(FFNClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.network(x)

# --- Helper Optimization Function ---
def train_and_evaluate_ffn(X_train, y_train, X_test, y_test, input_dim, experiment_name):
    """Handles PyTorch dataset setup, training loops, and evaluation logging."""
    wandb.init(project="NALAPRO-Project", name=experiment_name, reinit=True)

    # Convert numpy matrices to PyTorch Tensors
    train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
    test_dataset = TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test))

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = FFNClassifier(input_dim=input_dim).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Log hyperparameters
    wandb.config.update({"architecture": "2-Layer FFN", "lr": 0.001, "batch_size": 64})

    print(f"\nTraining {experiment_name} on {device}...")
    for epoch in range(15):
        model.train()
        running_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)

        # Validation evaluation
        model.eval()
        all_preds = []
        with torch.no_grad():
            for inputs, targets in test_loader:
                inputs = inputs.to(device)
                outputs = model(inputs)
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                all_preds.extend(preds)

        acc = accuracy_score(y_test, all_preds)
        f1 = f1_score(y_test, all_preds, average='macro')

        wandb.log({"epoch": epoch, "loss": epoch_loss, "val_accuracy": acc, "val_f1_macro": f1})
        print(f"Epoch {epoch+1}/15 | Loss: {epoch_loss:.4f} | Val Acc: {acc:.4f}")

    wandb.finish()
    return acc, f1

# --- Word2Vec Document Representation Aggregate Generator ---
def generate_doc_vectors(tokenized_texts, w2v_model, vector_size=100):
    vectors = []
    for tokens in tokenized_texts:
        valid_vectors = [w2v_model.wv[w] for w in tokens if w in w2v_model.wv]
        if valid_vectors:
            vectors.append(np.mean(valid_vectors, axis=0))
        else:
            vectors.append(np.zeros(vector_size))
    return np.array(vectors)

# --- Main Task Pipeline ---
def run_task1():
    data = load_and_preprocess_data()
    train_idx, test_idx = data["splits"]

    y_train = np.array(data["raw_labels"])[train_idx]
    y_test = np.array(data["raw_labels"])[test_idx]

    # 1. --- Word2Vec 1-Epoch vs Multi-Epoch Visualization ---
    print("\n--- Training Word2Vec Models ---")
    tokenized_train = [data["tokenized_texts"][i] for i in train_idx]

    w2v_1 = Word2Vec(sentences=tokenized_train, vector_size=100, window=5, min_count=2, workers=4, epochs=1)
    w2v_20 = Word2Vec(sentences=tokenized_train, vector_size=100, window=5, min_count=2, workers=4, epochs=20)

    # Select analytical clusters for report visualization
    target_words = ["space", "orbit", "nasa", "computer", "graphics", "software", "god", "atheism", "religion"]
    words_in_vocab = [w for w in target_words if w in w2v_1.wv and w in w2v_20.wv]

    # Build t-SNE Plots
    for model, name in [(w2v_1, "1_Epoch"), (w2v_20, "20_Epoch")]:
        vectors = np.array([model.wv[w] for w in words_in_vocab])
        tsne = TSNE(n_components=2, perplexity=len(words_in_vocab)-1, random_state=42, init='pca')
        coords = tsne.fit_transform(vectors)

        plt.figure(figsize=(8, 6))
        plt.scatter(coords[:, 0], coords[:, 1], edgecolors='k', c='cyan', s=100)
        for i, word in enumerate(words_in_vocab):
            plt.annotate(word, xy=(coords[i, 0], coords[i, 1]), xytext=(5, 2), textcoords='offset points')
        plt.title(f"Word2Vec t-SNE Comparison - {name}")
        plt.grid(True)
        os.makedirs("report", exist_ok=True)
        plt.savefig(f"report/tsne_{name}.png")
        plt.close()
    print("t-SNE Visualizations saved inside the report/ directory.")

    # Generate Mean Word2Vec Feature Vectors
    X_w2v_train = generate_doc_vectors([data["tokenized_texts"][i] for i in train_idx], w2v_20)
    X_w2v_test = generate_doc_vectors([data["tokenized_texts"][i] for i in test_idx], w2v_20)

    # Train Word2Vec FFN
    train_and_evaluate_ffn(X_w2v_train, y_train, X_w2v_test, y_test, 100, "Task_1b_Word2Vec_FFN")

    # 2. --- TF-IDF Vectorization & FFN ---
    print("\n--- Computing TF-IDF Embeddings ---")
    tfidf = TfidfVectorizer(max_features=2500)
    X_tfidf_all = tfidf.fit_transform(data["cleaned_texts"]).toarray()

    X_tfidf_train = X_tfidf_all[train_idx]
    X_tfidf_test = X_tfidf_all[test_idx]

    train_and_evaluate_ffn(X_tfidf_train, y_train, X_tfidf_test, y_test, 2500, "Task_1c_TFIDF_FFN")

    # 3. --- Custom Experiment: Hybrid Concatenation Matrix ---
    print("\n--- Running Custom Experiment: Concatenation Matrix ---")
    X_hybrid_train = np.concatenate([X_tfidf_train, X_w2v_train], axis=1)
    X_hybrid_test = np.concatenate([X_tfidf_test, X_w2v_test], axis=1)

    train_and_evaluate_ffn(X_hybrid_train, y_train, X_hybrid_test, y_test, 2600, "Task_1d_Custom_Hybrid_FFN")

if __name__ == "__main__":
    run_task1()

Loading 20Newsgroups data...
Preprocessing corpus...

--- Training Word2Vec Models ---
t-SNE Visualizations saved inside the report/ directory.



Training Task_1b_Word2Vec_FFN on cuda...
Epoch 1/15 | Loss: 1.8653 | Val Acc: 0.5589
Epoch 2/15 | Loss: 1.3962 | Val Acc: 0.5817
Epoch 3/15 | Loss: 1.3255 | Val Acc: 0.5966
Epoch 4/15 | Loss: 1.2777 | Val Acc: 0.6008
Epoch 5/15 | Loss: 1.2505 | Val Acc: 0.6019
Epoch 6/15 | Loss: 1.2280 | Val Acc: 0.6085
Epoch 7/15 | Loss: 1.2068 | Val Acc: 0.6117
Epoch 8/15 | Loss: 1.1882 | Val Acc: 0.6146
Epoch 9/15 | Loss: 1.1689 | Val Acc: 0.6180
Epoch 10/15 | Loss: 1.1617 | Val Acc: 0.6212
Epoch 11/15 | Loss: 1.1408 | Val Acc: 0.6207
Epoch 12/15 | Loss: 1.1335 | Val Acc: 0.6223
Epoch 13/15 | Loss: 1.1199 | Val Acc: 0.6255
Epoch 14/15 | Loss: 1.1061 | Val Acc: 0.6249
Epoch 15/15 | Loss: 1.0937 | Val Acc: 0.6194


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
loss,█▄▃▃▂▂▂▂▂▂▁▁▁▁▁
val_accuracy,▁▃▅▅▆▆▇▇▇█▇███▇
val_f1_macro,▁▅▅▆▆▆▇▇█▇█████
epoch,14
loss,1.09366
val_accuracy,0.61936
val_f1_macro,0.60301



--- Computing TF-IDF Embeddings ---



Training Task_1c_TFIDF_FFN on cuda...
Epoch 1/15 | Loss: 2.3516 | Val Acc: 0.6125
Epoch 2/15 | Loss: 1.2676 | Val Acc: 0.6565
Epoch 3/15 | Loss: 0.9753 | Val Acc: 0.6565
Epoch 4/15 | Loss: 0.8213 | Val Acc: 0.6565
Epoch 5/15 | Loss: 0.7081 | Val Acc: 0.6592
Epoch 6/15 | Loss: 0.6198 | Val Acc: 0.6573
Epoch 7/15 | Loss: 0.5440 | Val Acc: 0.6512
Epoch 8/15 | Loss: 0.4835 | Val Acc: 0.6477
Epoch 9/15 | Loss: 0.4284 | Val Acc: 0.6382
Epoch 10/15 | Loss: 0.3837 | Val Acc: 0.6401
Epoch 11/15 | Loss: 0.3475 | Val Acc: 0.6424
Epoch 12/15 | Loss: 0.3131 | Val Acc: 0.6366
Epoch 13/15 | Loss: 0.2868 | Val Acc: 0.6345
Epoch 14/15 | Loss: 0.2640 | Val Acc: 0.6329
Epoch 15/15 | Loss: 0.2417 | Val Acc: 0.6308


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
loss,█▄▃▃▃▂▂▂▂▁▁▁▁▁▁
val_accuracy,▁█████▇▆▅▅▅▅▄▄▄
val_f1_macro,▁▇▇▇██▇▇▆▆▆▆▆▆▅
epoch,14
loss,0.24174
val_accuracy,0.63077
val_f1_macro,0.6243



--- Running Custom Experiment: Concatenation Matrix ---



Training Task_1d_Custom_Hybrid_FFN on cuda...
Epoch 1/15 | Loss: 1.7932 | Val Acc: 0.5955
Epoch 2/15 | Loss: 1.2344 | Val Acc: 0.6387
Epoch 3/15 | Loss: 1.0742 | Val Acc: 0.6599
Epoch 4/15 | Loss: 0.9532 | Val Acc: 0.6660
Epoch 5/15 | Loss: 0.8525 | Val Acc: 0.6737
Epoch 6/15 | Loss: 0.7612 | Val Acc: 0.6796
Epoch 7/15 | Loss: 0.6817 | Val Acc: 0.6830
Epoch 8/15 | Loss: 0.6113 | Val Acc: 0.6756
Epoch 9/15 | Loss: 0.5469 | Val Acc: 0.6801
Epoch 10/15 | Loss: 0.4888 | Val Acc: 0.6724
Epoch 11/15 | Loss: 0.4377 | Val Acc: 0.6782
Epoch 12/15 | Loss: 0.3943 | Val Acc: 0.6735
Epoch 13/15 | Loss: 0.3515 | Val Acc: 0.6748
Epoch 14/15 | Loss: 0.3190 | Val Acc: 0.6719
Epoch 15/15 | Loss: 0.2897 | Val Acc: 0.6753


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
loss,█▅▅▄▄▃▃▂▂▂▂▁▁▁▁
val_accuracy,▁▄▆▇▇██▇█▇█▇▇▇▇
val_f1_macro,▁▄▆▆█████▇█▇▇██
epoch,14
loss,0.28974
val_accuracy,0.67533
val_f1_macro,0.66925


In [ ]:
import torch
import numpy as np
from transformers import BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score
import wandb

# Removed: from data_loader import load_and_preprocess_data

class NewsgroupDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average='macro')
    }

def run_task2(freeze_backbone=False):
    exp_name = "Task_2_BERT_Frozen" if freeze_backbone else "Task_2_BERT_Full"
    wandb.init(project="NALAPRO-Project", name=exp_name, reinit=True) # Reverted to reinit=True

    data = load_and_preprocess_data()
    train_idx, test_idx = data["splits"]

    train_texts = [data["cleaned_texts"][i] for i in train_idx]
    test_texts = [data["cleaned_texts"][i] for i in test_idx]
    y_train = [data["raw_labels"][i] for i in train_idx]
    y_test = [data["raw_labels"][i] for i in test_idx]

    print("Tokenizing data with BERT tokenizers...")
    tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

    train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=256)
    test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=256)

    train_dataset = NewsgroupDataset(train_encodings, y_train)
    test_dataset = NewsgroupDataset(test_encodings, y_test)

    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=20)

    if freeze_backbone:
        print("Freezing the BERT Transformer backbone feature layers...")
        for param in model.bert.parameters():
            param.requires_grad = False

    training_args = TrainingArguments(
        output_dir='./results_task2',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        warmup_steps=100,
        weight_decay=0.01,
        logging_dir='./logs',
        logging_steps=50,
        eval_strategy="epoch", # Corrected from 'evaluation_strategy'
        save_strategy="epoch",
        report_to="wandb",
        run_name=exp_name,
        learning_rate=2e-5 if not freeze_backbone else 1e-3,
        load_best_model_at_end=True,
        fp16=True # Added for mixed-precision training
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    eval_results = trainer.evaluate()
    print(f"\nFinal Evaluation Results ({exp_name}): {eval_results}")

    if not freeze_backbone:
        # Save baseline weights for comparison against task 3 later
        model.save_pretrained("./saved_models/vanilla_bert_classifier")

    wandb.finish()

if __name__ == "__main__":
    # Run full fine-tuning first
    run_task2(freeze_backbone=False)
    # Run frozen backbone experiment
    run_task2(freeze_backbone=True)

Loading 20Newsgroups data...
Preprocessing corpus...
Tokenizing data with BERT tokenizers...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Final Evaluation Results (Task_2_BERT_Frozen): {'eval_loss': 2.1320812702178955, 'eval_accuracy': 0.43395225464190984, 'eval_f1_macro': 0.39258069836417236, 'eval_runtime': 15.3158, 'eval_samples_per_second': 246.151, 'eval_steps_per_second': 7.704, 'epoch': 3.0}


eval/accuracy,▁▇██
eval/f1_macro,▁▇██
eval/loss,█▃▁▁
eval/runtime,▁▄▁█
eval/samples_per_second,█▅█▁
eval/steps_per_second,█▅█▁
train/epoch,▁▁▁▁▂▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇█████
train/global_step,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇█████
train/grad_norm,▄▆▇▃▅█▄▅▅▂▄▅▅▃▅▅▅▆▂▅▂▃▄▁▃▃▂▆▂▇▄▂▆▄▄▃▂▃▁▂
train/learning_rate,▄████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁
+1,...


In [ ]:
import torch
from transformers import (
    BertTokenizerFast,
    BertForMaskedLM,
    BertForSequenceClassification,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)
import wandb
# from task2_bert import NewsgroupDataset, compute_metrics # Removed this line
# NewsgroupDataset and compute_metrics are defined in hEQBm90zfwib and are globally available
# from data_loader import load_and_preprocess_data # This import is incorrect as the function is in the global scope

def run_task3_domain_adaptation():
    # Removed: wandb.login() # Added to ensure non-interactive login before wandb.init
    # --- STAGE 1: Masked Language Modeling Domain Adaptation ---
    wandb.init(project="NALAPRO-Project", name="Task_3_Stage1_MLM", reinit=True)

    data = load_and_preprocess_data()
    # Masked language modeling leverages all corpus variants without referencing text labels
    all_texts = data["cleaned_texts"]

    tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
    print("Tokenizing complete corpus for Masked Language Modeling...")

    encodings = tokenizer(all_texts, truncation=True, padding=True, max_length=256, return_special_tokens_mask=True)

    class MLMDataset(torch.utils.data.Dataset):
        def __init__(self, encodings): self.encodings = encodings
        def __getitem__(self, idx): return {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        def __len__(self): return len(self.encodings['input_ids'])

    mlm_dataset = MLMDataset(encodings)

    # Establish a dynamic data collator that enforces the 15% token mask criteria
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)
    model = BertForMaskedLM.from_pretrained('bert-base-uncased')

    mlm_args = TrainingArguments(
        output_dir='./results_mlm',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        logging_steps=50,
        save_steps=500,
        weight_decay=0.01,
        learning_rate=2e-5,
        report_to="wandb"
    )

    trainer = Trainer(
        model=model,
        args=mlm_args,
        train_dataset=mlm_dataset,
        data_collator=data_collator
    )

    print("Starting Phase 1 Self-Supervised MLM Domain Adaptation...")
    trainer.train()
    # Save the adapted language model weights
    trainer.save_model("./saved_models/domain_adapted_bert_mlm")
    wandb.finish()

def run_task3_classification():
    # Removed: wandb.login() # Added to ensure non-interactive login before wandb.init
    # --- STAGE 2: Downstream Supervised Classification Tuning ---
    wandb.init(project="NALAPRO-Project", name="Task_3_Stage2_MLM_Classifier", reinit=True)

    data = load_and_preprocess_data()
    train_idx, test_idx = data["splits"]

    tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

    train_encodings = tokenizer([data["cleaned_texts"][i] for i in train_idx], truncation=True, padding=True, max_length=256)
    test_encodings = tokenizer([data["cleaned_texts"][i] for i in test_idx], truncation=True, padding=True, max_length=256)

    train_dataset = NewsgroupDataset(train_encodings, [data["raw_labels"][i] for i in train_idx])
    test_dataset = NewsgroupDataset(test_encodings, [data["raw_labels"][i] for i in test_idx])

    # Critical Step: Load weights from the domain-adapted model saved in Stage 1
    print("Initializing Classification Head over Domain-Adapted Weights...")
    model = BertForSequenceClassification.from_pretrained("./saved_models/domain_adapted_bert_mlm", num_labels=20)

    cls_args = TrainingArguments(
        output_dir='./results_mlm_classifier',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        logging_steps=50,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        weight_decay=0.01,
        report_to="wandb",
        load_best_model_at_end=True
    )

    trainer = Trainer(
        model=model,
        args=cls_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()
    eval_res = trainer.evaluate()
    print(f"\nFinal Evaluation Matrix for MLM Adapted Classifier: {eval_res}")
    wandb.finish()

if __name__ == "__main__":
    run_task3_domain_adaptation()
    run_task3_classification()

NameError: name 'load_and_preprocess_data' is not defined

In [ ]:
import torch
from transformers import (
    BertTokenizerFast,
    BertForMaskedLM,
    BertForSequenceClassification,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)
import wandb
# from task2_bert import NewsgroupDataset, compute_metrics # Removed this line
# NewsgroupDataset and compute_metrics are defined in hEQBm90zfwib and are globally available
# from data_loader import load_and_preprocess_data # This import is incorrect as the function is in the global scope

def run_task3_domain_adaptation():
    # Removed: wandb.login() # Added to ensure non-interactive login before wandb.init
    # --- STAGE 1: Masked Language Modeling Domain Adaptation ---
    wandb.init(project="NALAPRO-Project", name="Task_3_Stage1_MLM", reinit=True)

    data = load_and_preprocess_data()
    # Masked language modeling leverages all corpus variants without referencing text labels
    all_texts = data["cleaned_texts"]

    tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
    print("Tokenizing complete corpus for Masked Language Modeling...")

    encodings = tokenizer(all_texts, truncation=True, padding=True, max_length=256, return_special_tokens_mask=True)

    class MLMDataset(torch.utils.data.Dataset):
        def __init__(self, encodings): self.encodings = encodings
        def __getitem__(self, idx): return {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        def __len__(self): return len(self.encodings['input_ids'])

    mlm_dataset = MLMDataset(encodings)

    # Establish a dynamic data collator that enforces the 15% token mask criteria
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)
    model = BertForMaskedLM.from_pretrained('bert-base-uncased')

    mlm_args = TrainingArguments(
        output_dir='./results_mlm',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        logging_steps=50,
        save_steps=500,
        weight_decay=0.01,
        learning_rate=2e-5,
        report_to="wandb"
    )

    trainer = Trainer(
        model=model,
        args=mlm_args,
        train_dataset=mlm_dataset,
        data_collator=data_collator
    )

    print("Starting Phase 1 Self-Supervised MLM Domain Adaptation...")
    trainer.train()
    # Save the adapted language model weights
    trainer.save_model("./saved_models/domain_adapted_bert_mlm")
    wandb.finish()

def run_task3_classification():
    # Removed: wandb.login() # Added to ensure non-interactive login before wandb.init
    # --- STAGE 2: Downstream Supervised Classification Tuning ---
    wandb.init(project="NALAPRO-Project", name="Task_3_Stage2_MLM_Classifier", reinit=True)

    data = load_and_preprocess_data()
    train_idx, test_idx = data["splits"]

    tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

    train_encodings = tokenizer([data["cleaned_texts"][i] for i in train_idx], truncation=True, padding=True, max_length=256)
    test_encodings = tokenizer([data["cleaned_texts"][i] for i in test_idx], truncation=True, padding=True, max_length=256)

    train_dataset = NewsgroupDataset(train_encodings, [data["raw_labels"][i] for i in train_idx])
    test_dataset = NewsgroupDataset(test_encodings, [data["raw_labels"][i] for i in test_idx])

    # Critical Step: Load weights from the domain-adapted model saved in Stage 1
    print("Initializing Classification Head over Domain-Adapted Weights...")
    model = BertForSequenceClassification.from_pretrained("./saved_models/domain_adapted_bert_mlm", num_labels=20)

    cls_args = TrainingArguments(
        output_dir='./results_mlm_classifier',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        logging_steps=50,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        weight_decay=0.01,
        report_to="wandb",
        load_best_model_at_end=True
    )

    trainer = Trainer(
        model=model,
        args=cls_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()
    eval_res = trainer.evaluate()
    print(f"\nFinal Evaluation Matrix for MLM Adapted Classifier: {eval_res}")
    wandb.finish()

if __name__ == "__main__":
    run_task3_domain_adaptation()
    run_task3_classification()

NameError: name 'load_and_preprocess_data' is not defined

In [ ]:
import torch
from transformers import (
    BertTokenizerFast,
    BertForMaskedLM,
    BertForSequenceClassification,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)
import wandb
# from task2_bert import NewsgroupDataset, compute_metrics # Removed this line
# NewsgroupDataset and compute_metrics are defined in hEQBm90zfwib and are globally available
# from data_loader import load_and_preprocess_data # This import is incorrect as the function is in the global scope

def run_task3_domain_adaptation():
    # Removed: wandb.login() # Added to ensure non-interactive login before wandb.init
    # --- STAGE 1: Masked Language Modeling Domain Adaptation ---
    wandb.init(project="NALAPRO-Project", name="Task_3_Stage1_MLM", reinit=True)

    data = load_and_preprocess_data()
    # Masked language modeling leverages all corpus variants without referencing text labels
    all_texts = data["cleaned_texts"]

    tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
    print("Tokenizing complete corpus for Masked Language Modeling...")

    encodings = tokenizer(all_texts, truncation=True, padding=True, max_length=256, return_special_tokens_mask=True)

    class MLMDataset(torch.utils.data.Dataset):
        def __init__(self, encodings): self.encodings = encodings
        def __getitem__(self, idx): return {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        def __len__(self): return len(self.encodings['input_ids'])

    mlm_dataset = MLMDataset(encodings)

    # Establish a dynamic data collator that enforces the 15% token mask criteria
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)
    model = BertForMaskedLM.from_pretrained('bert-base-uncased')

    mlm_args = TrainingArguments(
        output_dir='./results_mlm',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        logging_steps=50,
        save_steps=500,
        weight_decay=0.01,
        learning_rate=2e-5,
        report_to="wandb"
    )

    trainer = Trainer(
        model=model,
        args=mlm_args,
        train_dataset=mlm_dataset,
        data_collator=data_collator
    )

    print("Starting Phase 1 Self-Supervised MLM Domain Adaptation...")
    trainer.train()
    # Save the adapted language model weights
    trainer.save_model("./saved_models/domain_adapted_bert_mlm")
    wandb.finish()

def run_task3_classification():
    # Removed: wandb.login() # Added to ensure non-interactive login before wandb.init
    # --- STAGE 2: Downstream Supervised Classification Tuning ---
    wandb.init(project="NALAPRO-Project", name="Task_3_Stage2_MLM_Classifier", reinit=True)

    data = load_and_preprocess_data()
    train_idx, test_idx = data["splits"]

    tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

    train_encodings = tokenizer([data["cleaned_texts"][i] for i in train_idx], truncation=True, padding=True, max_length=256)
    test_encodings = tokenizer([data["cleaned_texts"][i] for i in test_idx], truncation=True, padding=True, max_length=256)

    train_dataset = NewsgroupDataset(train_encodings, [data["raw_labels"][i] for i in train_idx])
    test_dataset = NewsgroupDataset(test_encodings, [data["raw_labels"][i] for i in test_idx])

    # Critical Step: Load weights from the domain-adapted model saved in Stage 1
    print("Initializing Classification Head over Domain-Adapted Weights...")
    model = BertForSequenceClassification.from_pretrained("./saved_models/domain_adapted_bert_mlm", num_labels=20)

    cls_args = TrainingArguments(
        output_dir='./results_mlm_classifier',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        logging_steps=50,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        weight_decay=0.01,
        report_to="wandb",
        load_best_model_at_end=True
    )

    trainer = Trainer(
        model=model,
        args=cls_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()
    eval_res = trainer.evaluate()
    print(f"\nFinal Evaluation Matrix for MLM Adapted Classifier: {eval_res}")
    wandb.finish()

if __name__ == "__main__":
    run_task3_domain_adaptation()
    run_task3_classification()

NameError: name 'load_and_preprocess_data' is not defined

In [ ]:
import torch
from transformers import (
    BertTokenizerFast,
    BertForMaskedLM,
    BertForSequenceClassification,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)
import wandb
# from task2_bert import NewsgroupDataset, compute_metrics # Removed this line
# NewsgroupDataset and compute_metrics are defined in hEQBm90zfwib and are globally available
# from data_loader import load_and_preprocess_data # This import is incorrect as the function is in the global scope

def run_task3_domain_adaptation():
    # Removed: wandb.login() # Added to ensure non-interactive login before wandb.init
    # --- STAGE 1: Masked Language Modeling Domain Adaptation ---
    wandb.init(project="NALAPRO-Project", name="Task_3_Stage1_MLM", reinit=True)

    data = load_and_preprocess_data()
    # Masked language modeling leverages all corpus variants without referencing text labels
    all_texts = data["cleaned_texts"]

    tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
    print("Tokenizing complete corpus for Masked Language Modeling...")

    encodings = tokenizer(all_texts, truncation=True, padding=True, max_length=256, return_special_tokens_mask=True)

    class MLMDataset(torch.utils.data.Dataset):
        def __init__(self, encodings): self.encodings = encodings
        def __getitem__(self, idx): return {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        def __len__(self): return len(self.encodings['input_ids'])

    mlm_dataset = MLMDataset(encodings)

    # Establish a dynamic data collator that enforces the 15% token mask criteria
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)
    model = BertForMaskedLM.from_pretrained('bert-base-uncased')

    mlm_args = TrainingArguments(
        output_dir='./results_mlm',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        logging_steps=50,
        save_steps=500,
        weight_decay=0.01,
        learning_rate=2e-5,
        report_to="wandb"
    )

    trainer = Trainer(
        model=model,
        args=mlm_args,
        train_dataset=mlm_dataset,
        data_collator=data_collator
    )

    print("Starting Phase 1 Self-Supervised MLM Domain Adaptation...")
    trainer.train()
    # Save the adapted language model weights
    trainer.save_model("./saved_models/domain_adapted_bert_mlm")
    wandb.finish()

def run_task3_classification():
    # Removed: wandb.login() # Added to ensure non-interactive login before wandb.init
    # --- STAGE 2: Downstream Supervised Classification Tuning ---
    wandb.init(project="NALAPRO-Project", name="Task_3_Stage2_MLM_Classifier", reinit=True)

    data = load_and_preprocess_data()
    train_idx, test_idx = data["splits"]

    tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

    train_encodings = tokenizer([data["cleaned_texts"][i] for i in train_idx], truncation=True, padding=True, max_length=256)
    test_encodings = tokenizer([data["cleaned_texts"][i] for i in test_idx], truncation=True, padding=True, max_length=256)

    train_dataset = NewsgroupDataset(train_encodings, [data["raw_labels"][i] for i in train_idx])
    test_dataset = NewsgroupDataset(test_encodings, [data["raw_labels"][i] for i in test_idx])

    # Critical Step: Load weights from the domain-adapted model saved in Stage 1
    print("Initializing Classification Head over Domain-Adapted Weights...")
    model = BertForSequenceClassification.from_pretrained("./saved_models/domain_adapted_bert_mlm", num_labels=20)

    cls_args = TrainingArguments(
        output_dir='./results_mlm_classifier',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        logging_steps=50,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        weight_decay=0.01,
        report_to="wandb",
        load_best_model_at_end=True
    )

    trainer = Trainer(
        model=model,
        args=cls_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()
    eval_res = trainer.evaluate()
    print(f"\nFinal Evaluation Matrix for MLM Adapted Classifier: {eval_res}")
    wandb.finish()

if __name__ == "__main__":
    run_task3_domain_adaptation()
    run_task3_classification()

NameError: name 'load_and_preprocess_data' is not defined